# iSCORS deliverable — condensation axes (tidy)

Loads the video **once** (streaming, RAM-capped) at the top, then three sections:
1. **Paper replication** — gamma, CV², slope-3 condensation, y-intercept (+ optional Pearson vs the .mat Cond_map).
2. **Single-cell most-reproducible axis** — reliability-max (cross-half GEVD) and the X⊥Y residual.
3. **Multi-axis decomposition** — G(τ) GEVD axis 1/2 and de-nuisanced ICA comp 1/2.

Conventions (so results are reproducible):
- **ICA order/sign are arbitrary in FastICA** → fixed: sparse (high |kurtosis|) = comp 1, heavy tail positive.
- **De-nuisance basis = poly3 + radial vignetting**, identical wherever ICA is used.
- The temporal flat-field cancels in C(τ)/C(0), so g_norm is insensitive to it.


In [ ]:
# ── setup: clone/update repo from GitHub, mount Drive, install deps ──
import os, sys, subprocess
REPO = 'https://github.com/breezy90126/iscors-net.git'; BRANCH = 'claude/dazzling-newton-qz0ffj'
REPO_DIR = '/content/iscors-net'
try:
    from google.colab import drive; drive.mount('/content/drive', force_remount=False)
except Exception: pass
if os.path.isdir(REPO_DIR):
    for c in (['git', '-C', REPO_DIR, 'fetch', 'origin'], ['git', '-C', REPO_DIR, 'checkout', BRANCH],
              ['git', '-C', REPO_DIR, 'pull', 'origin', BRANCH]): subprocess.run(c, check=False)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO, REPO_DIR], check=False)
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)
subprocess.run(['pip', 'install', '-q', 'tifffile', 'scipy', 'gradio', 'scikit-learn', 'scikit-image'], check=False)
print('setup done:', os.getcwd())

# ── imports used throughout ──
import gc, zipfile, numpy as np, matplotlib.pyplot as plt, tifffile
from scipy.ndimage import zoom
from scipy.linalg import eigh as geigh
from scipy.stats import pearsonr, kurtosis
from sklearn.decomposition import FastICA
from utils.gpu_iscors_fit import streaming_acf, fit_gamma_alpha_batched


In [ ]:
# ── config (EDIT paths; same as the runner notebooks) ──
ZIP_PATH    = '/content/drive/MyDrive/iscors_test/large_file.zip'   # .zip or a .tif
VIDEO_FNAME = 'COBRI_rarw_video.tif'
EXTRACT_DIR = '/content/real_data'
MASK_PATH   = 'data/condensation_mask.tif'
MAT_FNAME   = 'Output_iSCORS_map.mat'        # optional, for the Cond_map validation
N_FRAMES = 5000; BIN = 2; SLOPE = 3.0; GAMMA_SCALE = 2.0
RECON_TAUS = (1, 2, 4, 8, 16, 32, 48, 64, 96, 128)
print('config ready')


In [ ]:
# ── loaders (proven full-load: chunk-read + bin → ~2GB binned array, RAM-safe) ──
import os, sys, gc, zipfile, numpy as np, tifffile
from scipy.ndimage import gaussian_filter, zoom
import importlib, utils.gpu_iscors_fit as _gf; importlib.reload(_gf)
from utils.gpu_iscors_fit import gpu_fit_maps, compute_density, compute_g_norm_torch

def load_video(path, fname=None, n_frames=2000, bin_factor=2):
    if str(path).lower().endswith('.zip'):
        ed = globals().get('EXTRACT_DIR', '/content/real_data'); os.makedirs(ed, exist_ok=True)
        def _find(root, name):
            for dp, _, fs in os.walk(root):
                if name in fs: return os.path.join(dp, name)
            return None
        vp = _find(ed, fname)
        if not vp:
            with zipfile.ZipFile(path) as z: z.extractall(ed)
            vp = _find(ed, fname)
        assert vp, f'{fname} not found in zip'
    else:
        vp = path
    H0, W0 = tifffile.imread(vp, key=0).shape
    Hb, Wb = H0 // bin_factor, W0 // bin_factor
    with tifffile.TiffFile(vp) as tf:
        try:    total = int(tf.series[0].shape[0])
        except Exception: total = len(tf.pages)
    n = min(n_frames, total); Hc, Wc = Hb * bin_factor, Wb * bin_factor
    raw = np.empty((n, Hb, Wb), np.float32)                 # store BINNED only (small)
    for s in range(0, n, 100):                              # chunk-read 100 frames at a time
        e = min(s + 100, n); ch = tifffile.imread(vp, key=range(s, e)).astype(np.float32)
        raw[s:e] = ch[:, :Hc, :Wc].reshape(e - s, Hb, bin_factor, Wb, bin_factor).mean((2, 4))
    print(f'[load] {os.path.basename(vp)}: {n}/{total} frames  binned {raw.shape}')
    return raw

def preprocess(raw):
    ff = raw / (np.median(raw, axis=0)[None] + 1e-10)      # temporal flat-field
    for t in range(len(ff)): ff[t] /= (gaussian_filter(ff[t], 4) + 1e-10)   # per-frame BG, in place
    return ff

def load_mask(shape):
    mk = np.asarray(tifffile.imread(MASK_PATH)).squeeze()
    if mk.ndim == 3: mk = mk[..., 0]
    if mk.shape != shape: mk = zoom(mk, (shape[0] / mk.shape[0], shape[1] / mk.shape[1]), order=0)
    bd = np.zeros(shape, bool); bd[0, :] = bd[-1, :] = bd[:, 0] = bd[:, -1] = True
    return mk == min(np.unique(mk), key=lambda z: (mk[bd] == z).mean())   # interior level = nucleus
print('loaders defined')


In [ ]:
# ── load ONCE (full) → all per-pixel maps for full + halves, then free the video ──
raw = load_video(ZIP_PATH, VIDEO_FNAME, N_FRAMES, BIN); vp = preprocess(raw); del raw; gc.collect()
h = len(vp) // 2
def _maps(seg):
    d, _ = compute_density(seg, min_cv=0.005)
    g, _c, _z = compute_g_norm_torch(seg, RECON_TAUS, norm='nor1', min_cv=0.005)
    f = gpu_fit_maps(seg, recon_taus=RECON_TAUS, n_components=1, global_alpha=True,
                     gamma_scale=GAMMA_SCALE, min_cv=0.005, n_steps=400, verbose=False)
    return f['gamma'].astype(np.float32), d.astype(np.float32), g.cpu().numpy().astype(np.float32)
gamma, dens, G = _maps(vp)                 # full
g1g, g1d, G1   = _maps(vp[:h])             # first half
g2g, g2d, G2   = _maps(vp[h:])             # second half
del vp; gc.collect()
try:
    import torch; torch.cuda.empty_cache()
except Exception: pass
nuc = load_mask(dens.shape) & np.isfinite(gamma)
_lg = lambda a: np.log10(np.clip(a, 1e-12, None))
X,  Y  = _lg(1 / np.clip(gamma, 1e-6, None)), _lg(dens)
X1, Y1 = _lg(1 / np.clip(g1g, 1e-6, None)), _lg(g1d)
X2, Y2 = _lg(1 / np.clip(g2g, 1e-6, None)), _lg(g2d)
print('loaded once (video freed). nucleus px:', int(nuc.sum()))


In [ ]:
# ── helpers (also register maps for the viewer) ──
m = nuc & np.isfinite(X) & np.isfinite(Y)
ALL_MAPS = {}
def _full(vec_on_m):
    z = np.full(nuc.shape, np.nan); z[m] = vec_on_m; return z
def _gevd(a1, a2):
    Cg = (a1.T @ a2) / a1.shape[0]; Cg = (Cg + Cg.T) / 2
    v, w = geigh(Cg, np.cov(np.vstack([a1, a2]).T)); o = np.argsort(v)[::-1]
    return v[o], w[:, o]
def show(items, title):
    n = len(items); fig, ax = plt.subplots(1, n, figsize=(4.2*n, 4.2)); ax = np.atleast_1d(ax)
    for a, (t, d, cm) in zip(ax, items):
        ALL_MAPS[t] = (d, cm)
        im = a.imshow(d, cmap=cm, vmin=np.nanpercentile(d, 2), vmax=np.nanpercentile(d, 98))
        a.set_title(t, fontsize=10); a.axis('off'); plt.colorbar(im, ax=a, fraction=0.046)
    fig.suptitle(title); plt.tight_layout(); plt.show()


In [ ]:
# ── Section 1 — paper replication ──
b3 = float((Y[m] - SLOPE * X[m]).mean())
cond = _full(((X + SLOPE * Y - SLOPE * b3) / (1 + SLOPE ** 2))[m])
show([('gamma (log)',                _full(_lg(np.clip(gamma, 1e-6, None))[m]), 'inferno'),
      ('CV2 (log)',                  _full(Y[m]),                               'inferno'),
      ('slope-3 condensation',       cond,                                      'inferno'),
      ('y-intercept density (Y-3X)', _full((Y - SLOPE * X)[m]),                 'inferno')],
     'Section 1 - paper replication')

# optional: Pearson of our slope-3 condensation vs the .mat Cond_map
def _find(name):
    for dp, _, fs in os.walk(EXTRACT_DIR):
        if name in fs: return os.path.join(dp, name)
mp = _find(MAT_FNAME)
if mp:
    try:
        import scipy.io; mat = scipy.io.loadmat(mp)
    except NotImplementedError:
        import h5py; mat = {k: np.array(v) for k, v in h5py.File(mp, 'r').items()}
    if 'Cond_map' in mat:
        cg = np.asarray(mat['Cond_map']).astype(float).squeeze()
        o = np.rot90(np.flipud(cg), -1)                     # MATLAB → Python orientation
        if o.shape != nuc.shape: o = zoom(o, (nuc.shape[0]/o.shape[0], nuc.shape[1]/o.shape[1]), order=1)
        ok = nuc & np.isfinite(o) & np.isfinite(cond)
        print('Pearson(our slope-3 condensation, .mat Cond_map) =', round(pearsonr(o[ok], cond[ok])[0], 3))
else:
    print('(.mat Cond_map not found under EXTRACT_DIR — skipping validation)')


In [ ]:
# ── Section 2 — single-cell most-reproducible axis ──
mC = nuc & np.isfinite(X1) & np.isfinite(Y1) & np.isfinite(X2) & np.isfinite(Y2)
mx, my, sx, sy = X[mC].mean(), Y[mC].mean(), X[mC].std(), Y[mC].std()
f1 = np.vstack([(X1[mC]-mx)/sx, (Y1[mC]-my)/sy]); f2 = np.vstack([(X2[mC]-mx)/sx, (Y2[mC]-my)/sy])
_, Wr = _gevd(f1, f2); vrel = Wr[:, 0]
relmax = _full((vrel[0]*(X-mx)/sx + vrel[1]*(Y-my)/sy)[m])
def _rel(vec):
    p1 = vec[0]*(X1[mC]-mx)/sx + vec[1]*(Y1[mC]-my)/sy; p2 = vec[0]*(X2[mC]-mx)/sx + vec[1]*(Y2[mC]-my)/sy
    return pearsonr(p1, p2)[0]
a, b = np.polyfit(Y[m], X[m], 1); xperp = _full((X-(a*Y+b))[m])
print('reliability-max axis r(halves) =', round(_rel(vrel), 3), '  raw CV2 r(halves) =', round(pearsonr(Y1[mC], Y2[mC])[0], 3))
show([('reliability-max axis', relmax, 'inferno'), ('X-perp-Y residual', xperp, 'coolwarm')],
     'Section 2 - single-cell reproducible axis')


In [ ]:
# ── Section 3 — multi-axis decomposition (G(tau) GEVD + de-nuisanced ICA) ──
feat = lambda g, yl: np.concatenate([g, yl[..., None]], -1)
F1, F2, Ff = feat(G1, Y1)[m], feat(G2, Y2)[m], feat(G, Y)[m]
mu = (F1.mean(0)+F2.mean(0))/2; sd = (F1.std(0)+F2.std(0))/2 + 1e-9
_, W = _gevd((F1-mu)/sd, (F2-mu)/sd); pj = ((Ff-mu)/sd) @ W
yy, xx = np.mgrid[0:nuc.shape[0], 0:nuc.shape[1]]
xa = (xx[m]-xx[m].mean())/xx[m].std(); ya = (yy[m]-yy[m].mean())/yy[m].std()
D = np.vstack([np.ones_like(xa), xa, ya, xa**2, ya**2, xa*ya, xa**3, ya**3, xa**2*ya, xa*ya**2,
               np.sqrt(xa**2+ya**2), xa**2+ya**2]).T              # poly3 + radial vignetting
Pinv = np.linalg.pinv(D); dn = lambda Fm: Fm - D @ (Pinv @ Fm)
_, W2 = _gevd((dn(F1)-mu)/sd, (dn(F2)-mu)/sd)
S = FastICA(2, random_state=0, whiten='unit-variance', max_iter=1000).fit_transform(((dn(Ff)-mu)/sd) @ W2[:, :2])
S = S[:, np.argsort([-abs(kurtosis(S[:, 0])), -abs(kurtosis(S[:, 1]))])]   # sparse component first
for j in range(2):
    if abs(S[:, j].min()) > abs(S[:, j].max()): S[:, j] = -S[:, j]         # heavy tail positive
show([('G(tau) GEVD axis 1', _full(pj[:, 0]), 'inferno'), ('G(tau) GEVD axis 2', _full(pj[:, 1]), 'inferno'),
      ('ICA comp 1 (condensate)', _full(S[:, 0]), 'inferno'), ('ICA comp 2', _full(S[:, 1]), 'inferno')],
     'Section 3 - multi-axis decomposition')


In [ ]:
# ── Section 3b (optional): remove the diagonal scan stripe at FEATURE level, then re-decompose ──
# A periodic stripe survives ICA (it is a frequency carrier, not an independent source) and ramp
# removal cannot touch it (ramp = smooth/low-freq only). Notch the stripe FREQUENCY out of every
# feature map (Fourier directional band-stop), then redo GEVD + ICA so the artifact never enters
# the subspace. Set the band/angle to where the stripe peaks ACTUALLY are in the 2D FFT.
def directional_bandstop(img, r_min, r_max, ang_deg=(40.0,), ang_width=20.0, transition=3.0):
    H, W = img.shape; cy, cx = H / 2.0, W / 2.0
    fillv = np.nanmean(img); a = np.where(np.isfinite(img), img, fillv) - fillv
    yy, xx = np.mgrid[0:H, 0:W]; r = np.hypot(yy - cy, xx - cx)
    th = np.degrees(np.arctan2(yy - cy, xx - cx)) % 180.0           # line orientation
    ann = np.where((r >= r_min) & (r <= r_max), 1.0,
                   np.exp(-(np.clip(r - r_max, 0, None) ** 2 + np.clip(r_min - r, 0, None) ** 2) / (2 * transition ** 2)))
    wed = np.zeros_like(th)
    for a0 in ang_deg:
        d = np.abs(th - (a0 % 180.0)); d = np.minimum(d, 180 - d)
        wed = np.maximum(wed, np.where(d <= ang_width, 1.0,
                         np.exp(-(d - ang_width) ** 2 / (2 * (transition + 1) ** 2))))
    F = np.fft.fftshift(np.fft.fft2(a)) * (1.0 - ann * wed)         # band-STOP
    return np.real(np.fft.ifft2(np.fft.ifftshift(F))) + fillv

# --- tune these to the stripe peak you see in the 2D FFT (radial-STD peaks at LOW r here) ---
R_MIN, R_MAX, ANG, ANG_W, TRANS = 60.0, 110.0, [40.0], 20.0, 3.0
def _destripe_stack(stk):
    out = stk.copy()
    for k in range(stk.shape[-1]):
        mp = np.where(nuc, stk[..., k], np.nan)
        out[..., k] = np.where(nuc, directional_bandstop(mp, R_MIN, R_MAX, ANG, ANG_W, TRANS), 0.0)
    return out
G_c, G1_c, G2_c = _destripe_stack(G), _destripe_stack(G1), _destripe_stack(G2)
Yc  = np.where(nuc, directional_bandstop(np.where(nuc, Y,  np.nan), R_MIN, R_MAX, ANG, ANG_W, TRANS), Y)
Y1c = np.where(nuc, directional_bandstop(np.where(nuc, Y1, np.nan), R_MIN, R_MAX, ANG, ANG_W, TRANS), Y1)
Y2c = np.where(nuc, directional_bandstop(np.where(nuc, Y2, np.nan), R_MIN, R_MAX, ANG, ANG_W, TRANS), Y2)

# re-decompose on the de-striped features (same conventions as Section 3)
feat = lambda g, yl: np.concatenate([g, yl[..., None]], -1)
F1, F2, Ff = feat(G1_c, Y1c)[m], feat(G2_c, Y2c)[m], feat(G_c, Yc)[m]
mu = (F1.mean(0) + F2.mean(0)) / 2; sd = (F1.std(0) + F2.std(0)) / 2 + 1e-9
yy, xx = np.mgrid[0:nuc.shape[0], 0:nuc.shape[1]]
xa = (xx[m] - xx[m].mean()) / xx[m].std(); ya = (yy[m] - yy[m].mean()) / yy[m].std()
D = np.vstack([np.ones_like(xa), xa, ya, xa**2, ya**2, xa*ya, xa**3, ya**3, xa**2*ya, xa*ya**2,
               np.sqrt(xa**2 + ya**2), xa**2 + ya**2]).T
Pinv = np.linalg.pinv(D); dn = lambda Fm: Fm - D @ (Pinv @ Fm)
_, W2 = _gevd((dn(F1) - mu) / sd, (dn(F2) - mu) / sd)
S = FastICA(2, random_state=0, whiten='unit-variance', max_iter=1000).fit_transform(((dn(Ff) - mu) / sd) @ W2[:, :2])
S = S[:, np.argsort([-abs(kurtosis(S[:, 0])), -abs(kurtosis(S[:, 1]))])]
for j in range(2):
    if abs(S[:, j].min()) > abs(S[:, j].max()): S[:, j] = -S[:, j]
show([('ICA comp 1 (de-striped)', _full(S[:, 0]), 'inferno'),
      ('ICA comp 2 (de-striped)', _full(S[:, 1]), 'inferno')],
     f'Section 3b - de-striped re-ICA (stripe removed: angle {ANG}deg, r in [{R_MIN:.0f},{R_MAX:.0f}])')

In [ ]:
# ── comparison grid: 5 rows × 2 cols (same method per row); left=coolwarm, right=viridis ──
# Run Sections 1-3 (and optionally 3b) first so ALL_MAPS is populated. Each map is z-scored over
# the nucleus and shown on a shared scale. GEVD/ICA rows live on the FULL-ACF feature plane
# (not pure CV²/1/D*); ICA row is the de-nuisanced (ramp-removed) result.
rows = [('gamma (log)',              'CV2 (log)'),
        ('slope-3 condensation',     'y-intercept density (Y-3X)'),
        ('reliability-max axis',     'X-perp-Y residual'),
        ('G(tau) GEVD axis 1',       'G(tau) GEVD axis 2'),
        ('ICA comp 1 (condensate)',  'ICA comp 2')]
row_note = ['', '', '', '  [full-ACF plane, not pure CV²/1/D*]', '  [de-nuisanced / ramp-removed]']
def _z(d):
    s = d[nuc]; return (d - np.nanmean(s)) / (np.nanstd(s) + 1e-9)
fig, ax = plt.subplots(5, 2, figsize=(8, 18))
for i, (left, right) in enumerate(rows):
    for j, (name, cmap) in enumerate([(left, 'coolwarm'), (right, 'viridis')]):
        a = ax[i, j]
        if name in ALL_MAPS:
            d = _z(ALL_MAPS[name][0])
            im = a.imshow(d, cmap=cmap, vmin=-2, vmax=3)
            plt.colorbar(im, ax=a, fraction=0.046)
            a.set_title(name + (row_note[i] if j == 0 else ''), fontsize=8)
        else:
            a.text(0.5, 0.5, f'{name}\n(run its section first)', ha='center', va='center', fontsize=8)
        a.axis('off')
fig.suptitle('Candidate axes — z-scored, left coolwarm / right viridis', fontsize=13)
plt.tight_layout(); plt.show()
if 'SAVE_DIR' in globals():
    import os; fig.savefig(os.path.join(SAVE_DIR, 'axes_grid_5x2.png'), dpi=120, bbox_inches='tight')

In [ ]:
# ── viewer: pick any computed map (no video reload) ──
import gradio as gr
def _render(name):
    d, cm = ALL_MAPS[name]
    fig, a = plt.subplots(figsize=(5, 5))
    im = a.imshow(d, cmap=cm, vmin=np.nanpercentile(d, 2), vmax=np.nanpercentile(d, 98))
    a.axis('off'); plt.colorbar(im, ax=a, fraction=0.046); return fig
with gr.Blocks() as demo:
    gr.Markdown('## iSCORS axes viewer (cached maps)')
    names = list(ALL_MAPS)
    dd = gr.Dropdown(names, value=names[0], label='map')
    out = gr.Plot()
    dd.change(_render, dd, out); demo.load(lambda: _render(names[0]), None, out)
demo.launch(share=False)
